In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

if os.path.exists('/workspace/data'):
    DATA_DIR      = Path('/workspace/data')
    WORKSPACE_DIR = Path('/workspace')
elif os.path.exists('../environment/data'):
    DATA_DIR      = Path('../environment/data')
    WORKSPACE_DIR = Path('..')
elif os.path.exists('environment/data'):
    DATA_DIR      = Path('environment/data')
    WORKSPACE_DIR = Path('.')
else:
    DATA_DIR      = Path('data')
    WORKSPACE_DIR = Path('.')

# DST spring-forward: 2024-03-10 02:00 ET → 03:00 ET
# In UTC that moment is 2024-03-10 07:00 UTC
DST_BOUNDARY_UTC = pd.Timestamp('2024-03-10 07:00:00')

In [ ]:
shifts       = pd.read_csv(DATA_DIR / 'shifts.csv',
                            parse_dates=['shift_start', 'shift_end'])
employees    = pd.read_csv(DATA_DIR / 'employees.csv')
depts        = pd.read_csv(DATA_DIR / 'departments.csv')
ot_policy    = pd.read_csv(DATA_DIR / 'ot_policy.csv')
reassignments = pd.read_csv(DATA_DIR / 'reassignments.csv')
budget       = pd.read_csv(DATA_DIR / 'weekly_budget.csv')

In [ ]:
# Trap 2: convert contractor hourly_rate from US cents to USD
employees = employees.copy()
mask = employees['employee_type'] == 'contractor'
employees.loc[mask, 'hourly_rate'] = employees.loc[mask, 'hourly_rate'] / 100.0

In [ ]:
# Trap 1: shift_end is stored in UTC; shift_start is US Eastern naive.
# Convert shift_end from UTC to ET before computing durations.
# Pre-DST (before 2024-03-10 07:00 UTC) offset is -5h (EST);
# post-DST offset is -4h (EDT).
off_hours = np.where(shifts['shift_end'] < DST_BOUNDARY_UTC, 5, 4)
shifts = shifts.copy()
shifts['shift_end_et'] = shifts['shift_end'] - pd.to_timedelta(off_hours, unit='h')

In [ ]:
def week_start_of(ts_series):
    return (ts_series - pd.to_timedelta(ts_series.dt.weekday, unit='D')).dt.normalize()

# Split shifts that cross a Monday boundary (using ET times).
start_wk = week_start_of(shifts['shift_start'])
end_wk   = week_start_of(shifts['shift_end_et'])
crosses  = start_wk != end_wk

normal = shifts[~crosses].copy()
normal['seg_start'] = normal['shift_start']
normal['seg_end']   = normal['shift_end_et']

xw = shifts[crosses].copy()
xw['boundary'] = week_start_of(xw['shift_end_et'])

seg1 = xw.copy(); seg1['seg_start'] = xw['shift_start'];  seg1['seg_end'] = xw['boundary']
seg2 = xw.copy(); seg2['seg_start'] = xw['boundary'];     seg2['seg_end'] = xw['shift_end_et']

segments = pd.concat([normal, seg1, seg2], ignore_index=True)
segments['hours']      = (segments['seg_end'] - segments['seg_start']).dt.total_seconds() / 3600
segments['week_start'] = week_start_of(segments['seg_start']).dt.strftime('%Y-%m-%d')

In [ ]:
weekly_hrs = (
    segments.groupby(['employee_id', 'week_start'])['hours']
    .sum().reset_index()
    .rename(columns={'hours': 'total_hours'})
)

# Attach department, rate, and dept-specific OT threshold (Trap 4)
weekly_hrs = weekly_hrs.merge(
    employees[['employee_id', 'department_id', 'hourly_rate']], on='employee_id'
)
weekly_hrs = weekly_hrs.merge(ot_policy, on='department_id')

weekly_hrs['regular_hours'] = np.minimum(weekly_hrs['total_hours'],
                                          weekly_hrs['weekly_ot_threshold'])
weekly_hrs['ot_hours']      = np.maximum(
    weekly_hrs['total_hours'] - weekly_hrs['weekly_ot_threshold'], 0.0
)
weekly_hrs['actual_cost']   = (
    weekly_hrs['regular_hours'] * weekly_hrs['hourly_rate']
    + weekly_hrs['ot_hours']    * weekly_hrs['hourly_rate'] * 1.5
)

In [ ]:
# Trap 3: apply department reassignments.
# Reassigned hours are billed to the receiving dept at the employee's base rate.
# The home dept absorbs any OT premium; its cost is reduced by reassigned_hours × base_rate.
rc = reassignments.merge(employees[['employee_id', 'hourly_rate']], on='employee_id')
rc['reass_cost'] = rc['reassigned_hours'] * rc['hourly_rate']

# Deduct from home dept
home_ded = (rc.groupby(['employee_id', 'week_start'])['reass_cost']
              .sum().reset_index()
              .rename(columns={'reass_cost': 'home_deduction'}))
weekly_hrs = weekly_hrs.merge(home_ded, on=['employee_id', 'week_start'], how='left')
weekly_hrs['home_deduction'] = weekly_hrs['home_deduction'].fillna(0.0)
weekly_hrs['home_cost']      = weekly_hrs['actual_cost'] - weekly_hrs['home_deduction']

In [ ]:
# Aggregate by department / week
home_agg = (
    weekly_hrs.groupby(['department_id', 'week_start'])
    .agg(actual_cost=('home_cost', 'sum'), ot_hours=('ot_hours', 'sum'))
    .reset_index()
)

recv_agg = (
    rc.groupby(['to_dept_id', 'week_start'])['reass_cost']
    .sum().reset_index()
    .rename(columns={'to_dept_id': 'department_id', 'reass_cost': 'recv_cost'})
)

dept_week = home_agg.merge(recv_agg, on=['department_id', 'week_start'], how='left')
dept_week['recv_cost']   = dept_week['recv_cost'].fillna(0.0)
dept_week['actual_cost'] = dept_week['actual_cost'] + dept_week['recv_cost']
dept_week = dept_week.drop(columns=['recv_cost'])

In [ ]:
budget = budget.copy()
budget['budgeted_cost'] = budget['budgeted_hours'] * budget['avg_hourly_rate']

report = budget.merge(dept_week, on=['department_id', 'week_start'])
report = report.merge(depts, on='department_id')
report['variance'] = report['actual_cost'] - report['budgeted_cost']

report = (
    report[['department_id', 'department_name', 'week_start',
             'budgeted_cost', 'actual_cost', 'variance']]
    .sort_values(['department_id', 'week_start'])
    .reset_index(drop=True)
)

In [ ]:
report.to_csv(WORKSPACE_DIR / 'payroll_variance_report.csv', index=False)

In [ ]:
total_budgeted_cost    = float(round(report['budgeted_cost'].sum(), 2))
total_actual_cost      = float(round(report['actual_cost'].sum(), 2))
total_variance         = float(round(total_actual_cost - total_budgeted_cost, 2))
total_overtime_hours   = float(round(dept_week['ot_hours'].sum(), 2))
over_budget_week_count = int((report['variance'] > 0).sum())